# HSI 지표 생성 실습 노트북

이 노트북은 기존 `src/00`, `src/04`, `src/05` 스크립트의 핵심 흐름을 실습용으로 정리한 파일입니다.

흐름은 다음과 같습니다.

1. 한국 ETF 가격 데이터 불러오기 및 품질 점검
2. 일별 수익률 기반 사건 등급 분류
3. 월별 사건 카운트 생성
4. 가격 기반 일별 지표 계산
5. 월말 기준 HSI 입력 데이터 생성
6. HSI 상태명과 점수 생성
7. 사건 캘린더 구간과 HSI 상태 연결

실습 결과는 원래 `output/` 폴더를 덮어쓰지 않고 `practice/output/`에 저장합니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "practice":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_PATH = PROJECT_DIR / "data" / "raw" / "korea_etf.csv"
EVENT_CALENDAR_PATH = PROJECT_DIR / "data" / "reference" / "event_calendar_us_kr.csv"
PRACTICE_OUTPUT_DIR = PROJECT_DIR / "practice" / "output"
PRACTICE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK_DAYS = 60
SMALL_Q = 0.60
LARGE_Q = 0.90

print(PROJECT_DIR)
print(RAW_PATH)
print(PRACTICE_OUTPUT_DIR)

## 1. 가격 데이터 불러오기 및 품질 점검

In [ ]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as error:
            last_error = error

    raise RuntimeError(f"CSV 파일을 읽지 못했습니다: {path}\n마지막 오류: {last_error}")


def normalize_ticker(col) -> str:
    text = str(col).strip()

    if text.lower().startswith("unnamed"):
        return text

    if text.endswith(".0"):
        text = text[:-2]

    if text.isdigit() and len(text) <= 6:
        return text.zfill(6)

    return text


def detect_date_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        if str(col).strip().lower() in ["date", "datetime", "날짜"]:
            return col
    return df.columns[0]


def load_and_clean_price(raw_path: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    raw = read_csv_safely(raw_path)
    date_col = detect_date_column(raw)

    raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
    raw = raw.dropna(subset=[date_col]).copy()
    raw = raw.sort_values(date_col).set_index(date_col)
    raw.columns = [normalize_ticker(c) for c in raw.columns]

    price = raw.apply(pd.to_numeric, errors="coerce").sort_index()

    quality_rows = []
    for ticker in price.columns:
        s = price[ticker]
        valid = s.dropna()
        first_valid = valid.index.min() if not valid.empty else pd.NaT
        last_valid = valid.index.max() if not valid.empty else pd.NaT
        valid_count = int(s.notna().sum())
        missing_count = int(s.isna().sum())
        missing_ratio = float(s.isna().mean())

        quality_rows.append({
            "Ticker": ticker,
            "FirstValidDate": first_valid.date().isoformat() if pd.notna(first_valid) else "",
            "LastValidDate": last_valid.date().isoformat() if pd.notna(last_valid) else "",
            "ValidCount": valid_count,
            "MissingCount": missing_count,
            "MissingRatio": round(missing_ratio, 4),
            "UseCandidate": valid_count >= 252 and missing_ratio <= 0.50,
        })

    quality = pd.DataFrame(quality_rows)
    candidate_tickers = quality.loc[quality["UseCandidate"], "Ticker"].tolist()
    common = price[candidate_tickers].dropna() if candidate_tickers else pd.DataFrame()

    summary = pd.DataFrame([{
        "RawPath": str(raw_path),
        "TotalRows": len(price),
        "TotalAssets": price.shape[1],
        "CandidateAssets": len(candidate_tickers),
        "CommonStartDate": common.index.min().date().isoformat() if not common.empty else "",
        "CommonEndDate": common.index.max().date().isoformat() if not common.empty else "",
        "CommonRowsAfterDropNA": len(common),
    }])

    return price, quality, summary

In [ ]:
price, data_quality, data_summary = load_and_clean_price(RAW_PATH)

price.to_csv(PRACTICE_OUTPUT_DIR / "korea_etf_price_clean_practice.csv", encoding="utf-8-sig")
data_quality.to_csv(PRACTICE_OUTPUT_DIR / "korea_etf_data_quality_practice.csv", index=False, encoding="utf-8-sig")
data_summary.to_csv(PRACTICE_OUTPUT_DIR / "korea_etf_data_summary_practice.csv", index=False, encoding="utf-8-sig")

display(data_summary)
display(data_quality.sort_values(["UseCandidate", "ValidCount"]).head(20))
price.head()

## 2. 일별 수익률 기반 사건 등급 분류

각 ETF의 일별 수익률을 과거 60거래일 절대수익률 분위수와 비교해 `small`, `medium`, `large` 사건으로 분류합니다. 기준 계산에는 당일 값을 넣지 않기 위해 `shift(1)`을 적용합니다.

In [ ]:
def classify_events(returns: pd.DataFrame) -> pd.DataFrame:
    abs_ret = returns.abs()
    q60 = abs_ret.rolling(LOOKBACK_DAYS, min_periods=LOOKBACK_DAYS).quantile(SMALL_Q).shift(1)
    q90 = abs_ret.rolling(LOOKBACK_DAYS, min_periods=LOOKBACK_DAYS).quantile(LARGE_Q).shift(1)

    rows = []
    for date in returns.index:
        for ticker in returns.columns:
            r = returns.at[date, ticker]
            if pd.isna(r):
                continue

            threshold_small = q60.at[date, ticker]
            threshold_large = q90.at[date, ticker]
            if pd.isna(threshold_small) or pd.isna(threshold_large):
                continue

            abs_value = abs(r)
            if abs_value <= threshold_small:
                size_label = "small"
            elif abs_value <= threshold_large:
                size_label = "medium"
            else:
                size_label = "large"

            if r > 0:
                direction = "up"
            elif r < 0:
                direction = "down"
            else:
                direction = "flat"

            rows.append({
                "Date": date,
                "Ticker": ticker,
                "Return": r,
                "AbsReturn": abs_value,
                "Q60": threshold_small,
                "Q90": threshold_large,
                "SizeLabel": size_label,
                "Direction": direction,
                "EventLabel": f"{size_label}_{direction}",
            })

    return pd.DataFrame(rows)


def make_monthly_counts(daily_events: pd.DataFrame) -> pd.DataFrame:
    df = daily_events.copy()
    df["Month"] = df["Date"].dt.to_period("M").astype(str)

    count_table = (
        df.groupby(["Month", "Ticker", "EventLabel"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    expected_cols = [
        "small_up", "small_down",
        "medium_up", "medium_down",
        "large_up", "large_down",
        "small_flat", "medium_flat", "large_flat",
    ]
    for col in expected_cols:
        if col not in count_table.columns:
            count_table[col] = 0

    count_table["risk_event_count"] = count_table["medium_down"] + 2 * count_table["large_down"]
    count_table["overheat_event_count"] = count_table["medium_up"] + 2 * count_table["large_up"]
    count_table["event_balance"] = count_table["overheat_event_count"] - count_table["risk_event_count"]

    return count_table.sort_values(["Month", "Ticker"])


In [ ]:
returns = price.pct_change()
daily_events = classify_events(returns)
monthly_counts = make_monthly_counts(daily_events)

daily_events.to_csv(PRACTICE_OUTPUT_DIR / "daily_event_labels_practice.csv", index=False, encoding="utf-8-sig")
monthly_counts.to_csv(PRACTICE_OUTPUT_DIR / "monthly_event_counts_practice.csv", index=False, encoding="utf-8-sig")

print(daily_events.shape, monthly_counts.shape)
display(daily_events.head())
display(monthly_counts.head())

## 3. 가격 기반 HSI 입력 지표 계산

HSI 상태 분류에는 사건 카운트뿐 아니라 추세, 모멘텀, 변동성, 상대강도 지표를 함께 사용합니다.

In [ ]:
def calculate_daily_indicators(price: pd.DataFrame) -> pd.DataFrame:
    returns = price.pct_change()

    ma20 = price.rolling(20, min_periods=20).mean()
    ma60 = price.rolling(60, min_periods=60).mean()
    ma20_gap = price / ma20 - 1
    ma60_gap = price / ma60 - 1

    vol20 = returns.rolling(20, min_periods=20).std() * np.sqrt(252)
    vol60 = returns.rolling(60, min_periods=60).std() * np.sqrt(252)

    ret21 = price.pct_change(21)
    ret63 = price.pct_change(63)
    universe_ret63 = ret63.mean(axis=1)
    rel_strength_63 = ret63.sub(universe_ret63, axis=0)
    ma20_gap_q90 = ma20_gap.rolling(252, min_periods=120).quantile(0.90).shift(1)

    frames = []
    for ticker in price.columns:
        frames.append(pd.DataFrame({
            "Date": price.index,
            "Ticker": ticker,
            "price": price[ticker].values,
            "daily_return": returns[ticker].values,
            "ret21": ret21[ticker].values,
            "ret63": ret63[ticker].values,
            "ma20_gap": ma20_gap[ticker].values,
            "ma60_gap": ma60_gap[ticker].values,
            "vol20": vol20[ticker].values,
            "vol60": vol60[ticker].values,
            "rel_strength_63": rel_strength_63[ticker].values,
            "ma20_gap_q90": ma20_gap_q90[ticker].values,
        }))

    daily = pd.concat(frames, ignore_index=True)
    daily["Month"] = daily["Date"].dt.to_period("M").astype(str)
    return daily


def make_month_end_indicators(daily: pd.DataFrame) -> pd.DataFrame:
    month_end = (
        daily.sort_values(["Ticker", "Date"])
        .dropna(subset=["price"])
        .groupby(["Month", "Ticker"], as_index=False)
        .tail(1)
        .copy()
    )

    month_end = month_end[[
        "Month", "Ticker", "Date", "price", "ret21", "ret63",
        "ma20_gap", "ma60_gap", "vol20", "vol60",
        "rel_strength_63", "ma20_gap_q90",
    ]].rename(columns={"Date": "MonthEndDate"})

    return month_end


def merge_indicators_and_events(month_end: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    merged = pd.merge(month_end, events, on=["Month", "Ticker"], how="left")
    event_cols = [
        "large_down", "large_up", "medium_down", "medium_up",
        "small_down", "small_up", "small_flat",
        "risk_event_count", "overheat_event_count", "event_balance",
    ]

    for col in event_cols:
        if col not in merged.columns:
            merged[col] = 0
        merged[col] = merged[col].fillna(0)

    return merged


In [ ]:
daily_indicators = calculate_daily_indicators(price)
month_end = make_month_end_indicators(daily_indicators)
hsi_inputs = merge_indicators_and_events(month_end, monthly_counts)

daily_indicators.to_csv(PRACTICE_OUTPUT_DIR / "daily_hsi_indicators_practice.csv", index=False, encoding="utf-8-sig")
hsi_inputs.to_csv(PRACTICE_OUTPUT_DIR / "monthly_hsi_inputs_practice.csv", index=False, encoding="utf-8-sig")

print(daily_indicators.shape, hsi_inputs.shape)
hsi_inputs.head()

## 4. HSI 상태명 및 점수 생성

아래 기준은 프로젝트의 1차 실험 기준입니다. 금융 이론에서 검증된 공식이라기보다, 시장 상태를 일관되게 해석하기 위한 규칙 기반 분류입니다.

In [ ]:
def classify_row(row: pd.Series) -> pd.Series:
    ret63 = row.get("ret63", np.nan)
    ma20_gap = row.get("ma20_gap", np.nan)
    ma60_gap = row.get("ma60_gap", np.nan)
    vol20 = row.get("vol20", np.nan)
    vol60 = row.get("vol60", np.nan)
    rel_strength = row.get("rel_strength_63", np.nan)
    ma20_gap_q90 = row.get("ma20_gap_q90", np.nan)

    large_down = row.get("large_down", 0)
    large_up = row.get("large_up", 0)
    medium_down = row.get("medium_down", 0)
    medium_up = row.get("medium_up", 0)
    risk_event_count = row.get("risk_event_count", 0)
    overheat_event_count = row.get("overheat_event_count", 0)

    vol_rising = pd.notna(vol20) and pd.notna(vol60) and vol20 > vol60
    trend_negative = pd.notna(ma20_gap) and pd.notna(ma60_gap) and ma20_gap < 0 and ma60_gap < 0
    trend_positive = pd.notna(ma20_gap) and pd.notna(ma60_gap) and ma20_gap > 0 and ma60_gap > 0
    momentum_negative = pd.notna(ret63) and ret63 < 0
    momentum_positive = pd.notna(ret63) and ret63 > 0
    relative_positive = pd.notna(rel_strength) and rel_strength > 0
    relative_negative = pd.notna(rel_strength) and rel_strength < 0

    gap_overheated = False
    if pd.notna(ma20_gap):
        if pd.notna(ma20_gap_q90):
            gap_overheated = ma20_gap > ma20_gap_q90
        else:
            gap_overheated = ma20_gap > 0.05

    high_vol_mixed = large_up >= 1 and large_down >= 1 and vol_rising

    risk_score = 0
    risk_score += 2 if large_down >= 2 else 0
    risk_score += 1 if medium_down >= 3 else 0
    risk_score += 1 if risk_event_count >= 4 else 0
    risk_score += 1 if momentum_negative else 0
    risk_score += 1 if trend_negative else 0
    risk_score += 1 if vol_rising else 0
    risk_score += 1 if relative_negative else 0

    overheat_score = 0
    overheat_score += 2 if large_up >= 2 else 0
    overheat_score += 1 if medium_up >= 3 else 0
    overheat_score += 1 if overheat_event_count >= 4 else 0
    overheat_score += 1 if momentum_positive else 0
    overheat_score += 1 if trend_positive else 0
    overheat_score += 1 if gap_overheated else 0
    overheat_score += 1 if vol_rising else 0

    recovery_score = 0
    recovery_score += 1 if momentum_positive else 0
    recovery_score += 1 if trend_positive else 0
    recovery_score += 1 if relative_positive else 0
    recovery_score += 1 if risk_event_count <= 1 else 0
    recovery_score += 1 if pd.notna(vol20) and pd.notna(vol60) and vol20 <= vol60 else 0

    if risk_score >= 5 and large_down >= 2:
        state = "강한 위험악화"
        reason = "큰 하락 사건과 위험 지표가 동시에 강하게 나타남"
    elif high_vol_mixed and risk_score >= 3 and overheat_score >= 3:
        state = "고변동성 혼합구간"
        reason = "큰 상승과 큰 하락이 함께 나타나고 단기 변동성이 확대됨"
    elif risk_score >= 4:
        state = "위험악화"
        reason = "하락 사건·약세 추세·변동성 확대 신호가 우세함"
    elif overheat_score >= 5 and vol_rising:
        state = "불안정 과열"
        reason = "큰 상승 사건이 많지만 변동성도 함께 확대됨"
    elif overheat_score >= 4:
        state = "과열 후보"
        reason = "상승 사건과 양호한 추세가 강하지만 과도한 상승 여부 확인 필요"
    elif recovery_score >= 4:
        state = "안정적 회복 후보"
        reason = "추세·상대강도·변동성 안정 신호가 함께 나타남"
    elif recovery_score >= 3:
        state = "회복 후보"
        reason = "일부 회복 신호가 나타나지만 아직 확정적이지 않음"
    else:
        state = "중립/혼조"
        reason = "위험악화·과열·회복 중 어느 한쪽이 뚜렷하지 않음"

    return pd.Series({
        "risk_score": risk_score,
        "overheat_score": overheat_score,
        "recovery_score": recovery_score,
        "vol_rising": vol_rising,
        "trend_negative": trend_negative,
        "trend_positive": trend_positive,
        "momentum_negative": momentum_negative,
        "momentum_positive": momentum_positive,
        "relative_positive": relative_positive,
        "relative_negative": relative_negative,
        "gap_overheated": gap_overheated,
        "high_vol_mixed": high_vol_mixed,
        "HSIStateLabel": state,
        "StateReason": reason,
    })


def classify_states(merged: pd.DataFrame) -> pd.DataFrame:
    state_cols = merged.apply(classify_row, axis=1)
    result = pd.concat([merged, state_cols], axis=1)
    result["hsi_direction_score_draft"] = result["risk_score"] + 0.5 * result["overheat_score"] - result["recovery_score"]
    return result


def make_state_summary(labels: pd.DataFrame) -> pd.DataFrame:
    return (
        labels.groupby("HSIStateLabel")
        .size()
        .reset_index(name="Count")
        .sort_values("Count", ascending=False)
    )


In [ ]:
hsi_labels = classify_states(hsi_inputs)
state_summary = make_state_summary(hsi_labels)

hsi_labels.to_csv(PRACTICE_OUTPUT_DIR / "monthly_hsi_state_labels_practice.csv", index=False, encoding="utf-8-sig")
state_summary.to_csv(PRACTICE_OUTPUT_DIR / "monthly_hsi_state_summary_practice.csv", index=False, encoding="utf-8-sig")

display(state_summary)
hsi_labels[["Month", "Ticker", "risk_score", "overheat_score", "recovery_score", "HSIStateLabel", "hsi_direction_score_draft"]].head(20)

## 5. 사건 캘린더와 HSI 상태 연결

사건 구간이 속한 월의 HSI 상태 분포를 확인합니다. 월별 리밸런싱 분석에서는 짧은 사건도 해당 월 전체와 연결해 봅니다.

In [ ]:
def load_event_calendar(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    cal = pd.read_csv(path, encoding="utf-8-sig")
    cal["StartDate"] = pd.to_datetime(cal["StartDate"], errors="coerce")
    cal["EndDate"] = pd.to_datetime(cal["EndDate"], errors="coerce")
    return cal


def check_event_period_states(labels: pd.DataFrame, event_calendar: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    distribution_rows = []
    temp = labels.copy()
    temp["Month"] = temp["Month"].astype(str)

    for _, event in event_calendar.iterrows():
        start = event["StartDate"]
        end = event["EndDate"]
        if pd.isna(start) or pd.isna(end):
            continue

        start_month = start.to_period("M").strftime("%Y-%m")
        end_month = end.to_period("M").strftime("%Y-%m")
        sub = temp[(temp["Month"] >= start_month) & (temp["Month"] <= end_month)].copy()

        if sub.empty:
            rows.append({
                "Market": event.get("Market", ""),
                "EventName": event.get("EventName", ""),
                "EventType": event.get("EventType", ""),
                "ExpectedHSIDirection": event.get("ExpectedHSIDirection", ""),
                "StartMonth": start_month,
                "EndMonth": end_month,
                "OverlapStatus": "no_overlap",
                "ObsCount": 0,
                "AssetCount": 0,
                "MeanRiskScore": np.nan,
                "MeanOverheatScore": np.nan,
                "MeanRecoveryScore": np.nan,
                "MostCommonState": "",
                "MostCommonStateCount": 0,
            })
            continue

        state_counts = sub["HSIStateLabel"].value_counts()
        rows.append({
            "Market": event.get("Market", ""),
            "EventName": event.get("EventName", ""),
            "EventType": event.get("EventType", ""),
            "ExpectedHSIDirection": event.get("ExpectedHSIDirection", ""),
            "StartMonth": start_month,
            "EndMonth": end_month,
            "OverlapStatus": "overlap",
            "ObsCount": len(sub),
            "AssetCount": sub["Ticker"].nunique(),
            "MeanRiskScore": sub["risk_score"].mean(),
            "MeanOverheatScore": sub["overheat_score"].mean(),
            "MeanRecoveryScore": sub["recovery_score"].mean(),
            "MostCommonState": state_counts.index[0],
            "MostCommonStateCount": int(state_counts.iloc[0]),
        })

        for state_name, count in state_counts.items():
            distribution_rows.append({
                "Market": event.get("Market", ""),
                "EventName": event.get("EventName", ""),
                "EventType": event.get("EventType", ""),
                "ExpectedHSIDirection": event.get("ExpectedHSIDirection", ""),
                "StartMonth": start_month,
                "EndMonth": end_month,
                "HSIStateLabel": state_name,
                "Count": int(count),
                "Share": float(count / len(sub)),
            })

    return pd.DataFrame(rows), pd.DataFrame(distribution_rows)


In [ ]:
event_calendar = load_event_calendar(EVENT_CALENDAR_PATH)
event_period_check, event_period_distribution = check_event_period_states(hsi_labels, event_calendar)

event_period_check.to_csv(PRACTICE_OUTPUT_DIR / "event_period_state_check_practice.csv", index=False, encoding="utf-8-sig")
event_period_distribution.to_csv(PRACTICE_OUTPUT_DIR / "event_period_state_distribution_practice.csv", index=False, encoding="utf-8-sig")

display(event_period_check)
display(event_period_distribution.head(20))

## 6. 생성된 실습 파일 확인

In [ ]:
for path in sorted(PRACTICE_OUTPUT_DIR.glob("*.csv")):
    print(path.name, f"{path.stat().st_size:,} bytes")